In [17]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

In [6]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### Download all the data from Qdrant

In [7]:
all_points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-00",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [12]:
all_points[0][0].payload

{'description': "HD Webcam - USB 2.0 Digital Video Webcamera for Computer PC Laptop Up & down 30 degrees rotatable, you can adjust the angle as you like. Support windows 2000/ XP/ Win7/ Win8/ Win10/ Vista 32bit./ Mac. Designed for both laptop and desktop, auto White balance, auto color correction. Support various video meeting software, ie, netmeeting and works great with msn, WeChat, QQ, Yahoo and Skype etc. High definition and true color images, manual focus. Built-in sound absorption Microphone, your voice can be heard clearly in 30 feet, meaning that you don't have to get close to even kiss Your camera awkwardly. Imported Optical lens, High precision and no distorted pictures, compatible with USB. The computer would automatically install the driver in the Win7 and lower level system after inserting this webcam, but in Win10, you don't need to install any driver, you can directly use the video chat software features and then can see the picture. []",
 'image': 'https://m.media-amazo

In [13]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["description"]} for data in all_points[0]]

In [14]:
all_context

[{'id': 'B01AVUAC5Q',
  'text': "HD Webcam - USB 2.0 Digital Video Webcamera for Computer PC Laptop Up & down 30 degrees rotatable, you can adjust the angle as you like. Support windows 2000/ XP/ Win7/ Win8/ Win10/ Vista 32bit./ Mac. Designed for both laptop and desktop, auto White balance, auto color correction. Support various video meeting software, ie, netmeeting and works great with msn, WeChat, QQ, Yahoo and Skype etc. High definition and true color images, manual focus. Built-in sound absorption Microphone, your voice can be heard clearly in 30 feet, meaning that you don't have to get close to even kiss Your camera awkwardly. Imported Optical lens, High precision and no distorted pictures, compatible with USB. The computer would automatically install the driver in the Win7 and lower level system after inserting this webcam, but in Win10, you don't need to install any driver, you can directly use the video chat software features and then can see the picture. []"},
 {'id': 'B09SBM

### Render a prompt to generate synthetic Eval reference dataset

In [23]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question."
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "items": {
                        "type": "string",
                        "description": "ID of the chunk that could be used to answer the question."
                    }
                }
            },
            "answer_example": {
                "type": "string",
                "description": "Suggest answer grounded in the context."
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks."
            }
        }
    }
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As a output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, Provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Also, include 5 questions that can't be answerd with the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
<OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{all_context}
"""

In [22]:
SYSTEM_PROMPT

'\nI am building a RAG application. I have a collection of 50 chunks of text.\nThe RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.\nI will provide all of the available products to you with IDs of each chunk.\nI want you to come up with 30 questions to which the answers could be grounded in the chunk context.\nThe questions should imitate a potential real user of this RAG system.\nAs a output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.\nAlso, Provide an example answer to the question given the context of the chunks.\nAlso, provide the reason why you chose the chunks to answer the questions.\nConstruct 10 that could use multiple chunks in the answer.\nConstruct 15 questions that could use single chunk in the answer.\nAlso, include 5 questions that can\'t be answerd with the available chunks.\n\n<OUTPUT JSON SCHEMA>\n{\n  "type": "array",\n  "items"

In [24]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text:
[{'id': 'B01AVUAC5Q', 'text': "HD Webcam - USB 2.0 Digital Video Webcamera for Computer PC Laptop Up & down 30 degrees rotatable, you can adjust the angle as you like. Support windows 2000/ XP/ Win7/ Win8/ Win10/ Vista 32bit./ Mac. Designed for both laptop and desktop, auto White balance, auto color correction. Support various video meeting software, ie, netmeeting and works great with msn, WeChat, QQ, Yahoo and Skype etc. High definition and true color images, manual focus. Built-in sound absorption Microphone, your voice can be heard clearly in 30 feet, meaning that you don't have to get close to even kiss Your camera awkwardly. Imported Optical lens, High precision and no distorted pictures, compatible with USB. The computer would automatically install the driver in the Win7 and lower level system after inserting this webcam, but in Win10, you don't need to install any driver, you can directly use the v

In [25]:
response = openai.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="low"
)

print(response.choices[0].message.content)

[
  {
    "question": "Which webcam would be suitable for Windows 10 video calls?",
    "chunk_ids": ["B01AVUAC5Q"],
    "answer_example": "The HD Webcam in chunk B01AVUAC5Q supports Windows 10, works with video-chat software, and does not require a driver installation on Windows 10.",
    "reasoning": "This chunk specifically describes Windows 10 compatibility, driver behavior, and video-chat support."
  },
  {
    "question": "Does the Articka drawing glove work for both left- and right-handed artists?",
    "chunk_ids": ["B09SBMZMFQ"],
    "answer_example": "Yes. The Articka two-finger drawing glove can be used on either the right or left hand.",
    "reasoning": "The product description explicitly states that the glove is suitable for both right and left hands."
  },
  {
    "question": "What health-tracking features does the TicWatch E3 have?",
    "chunk_ids": ["B0928CY9TN"],
    "answer_example": "The TicWatch E3 offers 24-hour heart-rate monitoring, sleep tracking, stress monit

In [27]:
json_output = response.choices[0].message.content
json_output = json.loads(json_output)

In [28]:
json_output

[{'question': 'Which webcam would be suitable for Windows 10 video calls?',
  'chunk_ids': ['B01AVUAC5Q'],
  'answer_example': 'The HD Webcam in chunk B01AVUAC5Q supports Windows 10, works with video-chat software, and does not require a driver installation on Windows 10.',
  'reasoning': 'This chunk specifically describes Windows 10 compatibility, driver behavior, and video-chat support.'},
 {'question': 'Does the Articka drawing glove work for both left- and right-handed artists?',
  'chunk_ids': ['B09SBMZMFQ'],
  'answer_example': 'Yes. The Articka two-finger drawing glove can be used on either the right or left hand.',
  'reasoning': 'The product description explicitly states that the glove is suitable for both right and left hands.'},
 {'question': 'What health-tracking features does the TicWatch E3 have?',
  'chunk_ids': ['B0928CY9TN'],
  'answer_example': 'The TicWatch E3 offers 24-hour heart-rate monitoring, sleep tracking, stress monitoring, blood oxygen saturation detection, 

In [29]:
len(json_output)

35

In [31]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-00",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B09SBMZMFQ")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [32]:
points[0].payload

{'description': "Articka Drawing Glove for Digital Drawing Tablet, iPad (Smudge Guard, Two-Finger, Reduces Friction, Elastic Lycra, Good for Right and Left Hand)(Large, Black) BONUS: Includes A FREE course of 24.99$ by 21draw - Learn to Draw Like a Pro. MATERIAL: Articka Glove has been designed with high-quality 100% elastic Lycra which is the best material for elasticity and for ensuring the maximum comfort. Soft, durable, light, and breathable. MOVE SMOOTH: Artickas two-finger drawing glove is designed for digital artists. Allowing the hand to slide smoothly over the tablet screen and eliminating the friction between the screen and the surface. This lightweight artist glove is made for any graphic tablet: iPad, Wacom, tracing light pad, lightbox, drawing monitors and digital panels. Making the process of sketching, inking, coloring, painting and drawing much more smooth and fluid. Eliminating the hand from sticking to the surface. DESIGN: The glove prevents smudges and eliminates the

In [33]:
def get_description(parent_asin: str) -> str:
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-00",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0]

    return points[0].payload["description"]

In [34]:
get_description("B09SBMZMFQ")

"Articka Drawing Glove for Digital Drawing Tablet, iPad (Smudge Guard, Two-Finger, Reduces Friction, Elastic Lycra, Good for Right and Left Hand)(Large, Black) BONUS: Includes A FREE course of 24.99$ by 21draw - Learn to Draw Like a Pro. MATERIAL: Articka Glove has been designed with high-quality 100% elastic Lycra which is the best material for elasticity and for ensuring the maximum comfort. Soft, durable, light, and breathable. MOVE SMOOTH: Artickas two-finger drawing glove is designed for digital artists. Allowing the hand to slide smoothly over the tablet screen and eliminating the friction between the screen and the surface. This lightweight artist glove is made for any graphic tablet: iPad, Wacom, tracing light pad, lightbox, drawing monitors and digital panels. Making the process of sketching, inking, coloring, painting and drawing much more smooth and fluid. Eliminating the hand from sticking to the surface. DESIGN: The glove prevents smudges and eliminates the oils that appea

### Create Eval dataset in langsmith

In [35]:
client = Client(api_key=os.environ["LANGSMITH_API_KEY"])

In [36]:
dataset_name = "rag-evaluation-dataset"
dataset = client.create_dataset(
    dataset_name=dataset_name, 
    description="Dataset for evaluating RAG pipeline"
)

In [40]:
for item in json_output:
    client.create_example(
        dataset_id=dataset.id, 
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )